# Phase 1 — System Boundaries, Data Dictionary & Engineering Assumptions

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Provisional Pilot Site:** Settat, Casablanca-Settat, Morocco (`33.0010° N, -7.6166° E`)

## Learning Objectives
1. Establish clear thermodynamic control volumes separating electrical generation (`kW_e / kWh_e`) from thermal heat flows (`kW_th / kWh_th`).
2. Inspect the 20-variable schema and its four data provenance classes (`measured_external`, `simulated`, `calculated`, `predicted_ml`).
3. Verify baseline pilot-plant design assumptions (`PlantDesignAssumptions`) and hardwired PLC safety limits (`PLC_HARDWIRED_INTERLOCKS`) that AI can never override.
4. Export structured Phase 1 parameters, variable schema, and missing-plant-information tracking tables to UTF-8 CSV files.

## Core Sizing Equations
- **Usable Sand Storage Capacity:** $Q_{\text{stored}} = m_{\text{sand}} \cdot C_{p,\text{sand}} \cdot (T_{\text{max}} - T_{\text{min}}) / 3600$ `[kWh_th]`
- **Water Evaporation Mass Balance (Wet Basis):** $m_{\text{evap}} = m_{\text{wet}} \cdot \frac{X_i - X_f}{1 - X_f}$ `[kg]`


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Using Colab built-in scientific Python stack (pandas, numpy, scikit-learn, matplotlib, seaborn).')

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'src/config.py',
    'docs/phase1_system_boundaries.md',
    'requirements.txt',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import pandas as pd
from dataclasses import asdict
from src.config import PlantDesignAssumptions, PLC_HARDWIRED_INTERLOCKS, VARIABLE_SCHEMA

assumptions = PlantDesignAssumptions()
print("Provisional Site:", assumptions.site_name)
print(f"PV Ref Capacity : {assumptions.pv_nominal_capacity_kwp} kWp (tilt={assumptions.pv_tilt_deg} deg)")
print(f"Sand Storage    : {assumptions.sand_mass_kg} kg, {assumptions.sand_temp_min_operating_c}-{assumptions.sand_temp_max_operating_c} deg C (PLC trip {assumptions.sand_temp_plc_trip_c} deg C)")

# Thermodynamic boundary check
delta_t = assumptions.sand_temp_max_operating_c - assumptions.sand_temp_min_operating_c
usable_kwh = (assumptions.sand_mass_kg * assumptions.sand_specific_heat_kj_per_kg_k * delta_t) / 3600.0
assert assumptions.sand_temp_max_operating_c < assumptions.sand_temp_plc_trip_c, "Safety margin violated!"
print(f"Nominal Usable Thermal Storage Capacity: {usable_kwh:.2f} kWh_th")

print("\nPLC Interlocks:")
for plc in PLC_HARDWIRED_INTERLOCKS:
    print(f"  - {plc['id']} ({plc['subsystem']}): {plc['interlock']}")


In [ ]:
# Inspect & Export Phase 1 Structured CSV Tables
out_dir = PROJECT_ROOT / 'data' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

schema_rows = [
    {
        'variable_name': v.name,
        'label': v.label,
        'unit': v.unit,
        'subsystem': v.subsystem.value,
        'provenance': v.provenance.value,
        'min_valid': v.min_valid,
        'max_valid': v.max_valid,
        'source': v.source,
        'ml_role': 'feature' if v.used_as_ml_feature else (v.ml_target_for or 'process_state'),
        'description': v.description,
    }
    for v in VARIABLE_SCHEMA.values()
]
df_schema = pd.DataFrame(schema_rows)
df_schema.to_csv(out_dir / 'phase1_variable_schema.csv', index=False, encoding='utf-8')
display(df_schema[['variable_name', 'unit', 'subsystem', 'provenance', 'ml_role']].head(10))


## Conclusions & Limitations
- All site coordinates and equipment capacities (`15 kWp`, `4,000 kg` sand, `250 kg/batch`) are explicit provisional engineering assumptions until pilot plant commissioning measurements become available.
- Electrical and thermal energy quantities remain strictly separate throughout all subsequent phases.


In [ ]:
# Export & Download Phase 1 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_01'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase01_Package.zip'

patterns = ['README.md', 'requirements.txt', 'DATA_DICTIONARY.csv', 'PROJECT_MANIFEST.csv', 'src/config.py', 'docs/phase1_system_boundaries.md', 'data/outputs/phase1_*.csv', 'notebooks/00_phase1_project_definition.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
